In [1]:
"""
Toy GCN Water Leakage Detector
================================
A miniature but mechanically legit implementation of the GCN approach
from "Detection and localization of pipeline leakage of water distribution
networks based on graph convolutional networks" (Li et al., 2025).

Network topology: 9 nodes, 12 pipes — a simple grid city.
Features per node: [pressure (m), water demand (L/s)]
Label: 0 = normal, 1 = leakage somewhere in the network

What's faithful to the paper:
  - Two-layer GCN with the symmetric normalised adjacency:  D^{-1/2} A~ D^{-1/2} H W
  - Leaky ReLU activations (negative_slope=0.01)
  - Global mean pooling  →  binary classifier
  - Cross-entropy loss, Adam optimiser (lr=0.001)
  - Gradient-based node importance: S_i = Σ_k |∂o_leak / ∂x_{i,k}|
  - Linear interpolation of node importance along edges for "risk contour"

Run:
    python gcn_leakage_demo.py
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import random

# ─────────────────────────────────────────────
# 1.  NETWORK TOPOLOGY  (9 nodes, 12 pipes)
#     Laid out as a 3×3 grid:
#
#       0 ─── 1 ─── 2
#       │     │     │
#       3 ─── 4 ─── 5
#       │     │     │
#       6 ─── 7 ─── 8
#
# ─────────────────────────────────────────────
EDGES = [
    (0,1),(1,2),          # top row
    (3,4),(4,5),          # mid row
    (6,7),(7,8),          # bot row
    (0,3),(3,6),          # left col
    (1,4),(4,7),          # mid col
    (2,5),(5,8),          # right col
]
N_NODES  = 9
N_EDGES  = len(EDGES)

# Node (x,y) positions for plotting
NODE_POS = {
    0:(0,2), 1:(1,2), 2:(2,2),
    3:(0,1), 4:(1,1), 5:(2,1),
    6:(0,0), 7:(1,0), 8:(2,0),
}

# Build COO edge_index (undirected → both directions)
_src = [u for u,v in EDGES] + [v for u,v in EDGES]
_dst = [v for u,v in EDGES] + [u for u,v in EDGES]
EDGE_INDEX = torch.tensor([_src, _dst], dtype=torch.long)

# ─────────────────────────────────────────────
# 2.  SYNTHETIC DATA GENERATOR
# ─────────────────────────────────────────────
BASE_PRESSURE = 50.0   # m   — nominal pressure at every node
BASE_DEMAND   = 1.0    # L/s — nominal demand

def make_graph(leaking_nodes=None, noise_std=0.5):
    """
    leaking_nodes: list of node indices that are 'leaking', or None for normal.
    Returns a torch_geometric Data object.
    """
    pressure = np.full(N_NODES, BASE_PRESSURE)
    demand   = np.full(N_NODES, BASE_DEMAND)

    if leaking_nodes:
        for ln in leaking_nodes:
            # Physics: pressure drop is strongest at the leak and decays with
            # graph distance (Laplacian smoothing analogy from the paper).
            for node in range(N_NODES):
                # rough BFS distance on the grid
                dist = abs((node % 3) - (ln % 3)) + abs((node // 3) - (ln // 3))
                drop = 12.0 * np.exp(-0.7 * dist)          # exponential decay
                pressure[node] -= drop
                demand[node]   += 0.3 * np.exp(-0.9 * dist) # demand spike near leak

    # Add measurement noise
    pressure += np.random.normal(0, noise_std, N_NODES)
    demand   += np.random.normal(0, noise_std * 0.1, N_NODES)

    x     = torch.tensor(np.stack([pressure, demand], axis=1), dtype=torch.float)
    label = 1 if leaking_nodes else 0
    y     = torch.tensor([label], dtype=torch.long)

    return Data(x=x, edge_index=EDGE_INDEX, y=y)


def build_dataset(n_normal=600, n_leak=600):
    graphs = []
    # Normal samples
    for _ in range(n_normal):
        graphs.append(make_graph(leaking_nodes=None))

    # Leakage samples — pick 1 or 2 random leaking nodes
    for _ in range(n_leak):
        k = random.randint(1, 2)
        leaking = random.sample(range(N_NODES), k)
        graphs.append(make_graph(leaking_nodes=leaking))

    random.shuffle(graphs)
    return graphs


# ─────────────────────────────────────────────
# 3.  GCN MODEL  (paper-faithful architecture)
# ─────────────────────────────────────────────
class WaterLeakGCN(nn.Module):
    def __init__(self, in_channels=2, hidden=32, out_channels=2):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)   # GCNConv implements D^{-1/2} A~ D^{-1/2} H W
        self.conv2 = GCNConv(hidden, hidden)
        self.classifier = nn.Linear(hidden, out_channels)
        self.act = nn.LeakyReLU(negative_slope=0.01)

    def forward(self, x, edge_index, batch):
        # Layer 1 + Leaky ReLU
        h = self.act(self.conv1(x, edge_index))
        # Layer 2
        h = self.conv2(h, edge_index)
        # Global mean pool → graph-level representation
        h_graph = global_mean_pool(h, batch)
        # Classification logits
        out = self.classifier(h_graph)
        return out, h          # also return node embeddings for importance


# ─────────────────────────────────────────────
# 4.  TRAINING
# ─────────────────────────────────────────────
def train_model(model, loader, n_epochs=40):
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001,
                                 betas=(0.9, 0.999), eps=1e-8)
    criterion = nn.CrossEntropyLoss()

    losses, accs = [], []
    model.train()
    for epoch in range(1, n_epochs + 1):
        total_loss, correct, total = 0.0, 0, 0
        for batch in loader:
            optimiser.zero_grad()
            out, _ = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y)
            loss.backward()
            # Gradient norm clip (paper section 2.4.1 step H)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()

            total_loss += loss.item() * batch.num_graphs
            pred   = out.argmax(dim=1)
            correct += (pred == batch.y).sum().item()
            total  += batch.num_graphs

        avg_loss = total_loss / total
        acc      = correct / total
        losses.append(avg_loss)
        accs.append(acc)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | CE loss: {avg_loss:.4f} | ACC: {acc*100:.1f}%")

        # Early stop if converged (paper criterion: CE < 0.01)
        if avg_loss < 0.01:
            print(f"  ✓ Converged at epoch {epoch}!")
            break

    return losses, accs


# ─────────────────────────────────────────────
# 5.  NODE IMPORTANCE  (paper eq. 6 & 7)
# ─────────────────────────────────────────────
def compute_node_importance(model, graph):
    """
    Returns S_i = Σ_k |∂o_leak / ∂x_{i,k}| for each node i.
    (paper eq. 7 — gradient of the leakage logit w.r.t. node features)
    """
    model.eval()
    x = graph.x.clone().detach().requires_grad_(True)
    batch = torch.zeros(N_NODES, dtype=torch.long)

    out, _ = model(x, graph.edge_index, batch)
    o_leak = out[0, 1]   # logit for class "leakage"
    o_leak.backward()

    # S_i = sum over feature dimensions of |gradient|
    importance = x.grad.abs().sum(dim=1).detach().numpy()
    return importance


def edge_risk(importance):
    """
    Paper eq. 8: linear interpolation of node importance along each edge.
    Returns a dict {(u,v): mean_risk} for plotting.
    """
    risks = {}
    for u, v in EDGES:
        # midpoint interpolation (τ=0.5)
        risks[(u, v)] = 0.5 * importance[u] + 0.5 * importance[v]
    return risks


# ─────────────────────────────────────────────
# 6.  DEMO SCENARIO  (the "button click")
# ─────────────────────────────────────────────
def run_demo(model, leaking_nodes=None, noise_std=1.0):
    """
    Simulates one live reading and runs detection + localisation.
    leaking_nodes: list of ints, or None for normal state.
    """
    graph = make_graph(leaking_nodes=leaking_nodes, noise_std=noise_std)

    # ── Detection ──────────────────────────────
    model.eval()
    with torch.no_grad():
        batch = torch.zeros(N_NODES, dtype=torch.long)
        out, _ = model(graph.x, graph.edge_index, batch)
        probs = F.softmax(out, dim=1)[0]
        pred  = out.argmax(dim=1).item()

    p_normal = probs[0].item()
    p_leak   = probs[1].item()
    state    = "⚠️  LEAKAGE DETECTED" if pred == 1 else "✅  NORMAL OPERATION"

    # ── Localisation ───────────────────────────
    importance = compute_node_importance(model, graph)
    risks      = edge_risk(importance)

    # ── Console output ─────────────────────────
    print("\n" + "="*55)
    print(f"  SENSOR READING RESULT")
    print("="*55)
    print(f"  Ground truth : {'LEAKAGE at nodes ' + str(leaking_nodes) if leaking_nodes else 'NORMAL'}")
    print(f"  Prediction   : {state}")
    print(f"  P(normal)    : {p_normal*100:.1f}%")
    print(f"  P(leakage)   : {p_leak*100:.1f}%")
    print("\n  Node importance scores (higher = more suspicious):")
    ranked = sorted(enumerate(importance), key=lambda x: -x[1])
    for rank, (node, score) in enumerate(ranked, 1):
        bar = "█" * int(score / importance.max() * 20)
        print(f"    #{rank}  Node {node}  {bar} {score:.3f}")
    print("="*55)

    # ── Plot ───────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor("#0f0f1a")

    # --- Left: pressure readings ---
    ax = axes[0]
    ax.set_facecolor("#0f0f1a")
    ax.set_title("Live Sensor Readings", color="white", fontsize=13, fontweight="bold", pad=12)

    pressures = graph.x[:, 0].numpy()
    demands   = graph.x[:, 1].numpy()
    G = nx.Graph()
    G.add_nodes_from(range(N_NODES))
    G.add_edges_from(EDGES)

    # Node colours by pressure (blue=high, red=low)
    p_norm = (pressures - pressures.min()) / (pressures.max() - pressures.min() + 1e-8)
    node_colors = plt.cm.RdYlBu(p_norm)

    nx.draw_networkx_edges(G, NODE_POS, ax=ax,
                           edge_color="#444466", width=2.0, alpha=0.8)
    nx.draw_networkx_nodes(G, NODE_POS, ax=ax,
                           node_color=node_colors, node_size=900, alpha=1.0)
    labels = {i: f"{i}\n{pressures[i]:.1f}m" for i in range(N_NODES)}
    nx.draw_networkx_labels(G, NODE_POS, labels=labels, ax=ax,
                            font_size=7, font_color="white", font_weight="bold")

    if leaking_nodes:
        nx.draw_networkx_nodes(G, NODE_POS, nodelist=leaking_nodes, ax=ax,
                               node_color="red", node_size=1100, alpha=0.5,
                               label="True leak")

    sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlBu,
                                norm=plt.Normalize(pressures.min(), pressures.max()))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.04, pad=0.04)
    cbar.set_label("Pressure (m)", color="white", fontsize=9)
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

    result_color = "#ff4444" if pred == 1 else "#44ff88"
    ax.text(0.5, -0.08, state, transform=ax.transAxes,
            ha="center", fontsize=11, color=result_color, fontweight="bold")
    ax.text(0.5, -0.14,
            f"P(leakage) = {p_leak*100:.1f}%   P(normal) = {p_normal*100:.1f}%",
            transform=ax.transAxes, ha="center", fontsize=9, color="#aaaacc")
    ax.axis("off")

    # --- Right: leakage risk map ---
    ax = axes[1]
    ax.set_facecolor("#0f0f1a")
    ax.set_title("GCN Leakage Risk Map  (gradient-based node importance)",
                 color="white", fontsize=11, fontweight="bold", pad=12)

    max_risk = max(risks.values()) if risks else 1.0
    for (u, v), risk in risks.items():
        color = plt.cm.hot(risk / max_risk)
        x0, y0 = NODE_POS[u]
        x1, y1 = NODE_POS[v]
        ax.plot([x0, x1], [y0, y1], color=color,
                linewidth=4 + 8 * (risk / max_risk), alpha=0.85, solid_capstyle="round")

    imp_norm = (importance - importance.min()) / (importance.max() - importance.min() + 1e-8)
    node_colors2 = plt.cm.hot(imp_norm)
    for i, (xpos, ypos) in NODE_POS.items():
        circle = plt.Circle((xpos, ypos), 0.18, color=node_colors2[i],
                             zorder=5, linewidth=2,
                             ec="white" if imp_norm[i] > 0.7 else "#555566")
        ax.add_patch(circle)
        ax.text(xpos, ypos, str(i), ha="center", va="center",
                fontsize=9, color="white", fontweight="bold", zorder=6)

    # Importance score labels
    top3 = sorted(range(N_NODES), key=lambda i: -importance[i])[:3]
    for i in top3:
        xpos, ypos = NODE_POS[i]
        ax.text(xpos + 0.22, ypos + 0.22,
                f"#{list(top3).index(i)+1}\n{importance[i]:.2f}",
                fontsize=7, color="#ffcc44", fontweight="bold", zorder=7)

    sm2 = plt.cm.ScalarMappable(cmap=plt.cm.hot,
                                  norm=plt.Normalize(importance.min(), importance.max()))
    sm2.set_array([])
    cbar2 = plt.colorbar(sm2, ax=ax, fraction=0.04, pad=0.04)
    cbar2.set_label("Node importance", color="white", fontsize=9)
    cbar2.ax.yaxis.set_tick_params(color="white")
    plt.setp(cbar2.ax.yaxis.get_ticklabels(), color="white")

    ax.set_xlim(-0.4, 2.6)
    ax.set_ylim(-0.4, 2.5)
    ax.set_aspect("equal")
    ax.axis("off")

    truth_str = f"nodes {leaking_nodes}" if leaking_nodes else "none (normal)"
    fig.suptitle(f"Water Distribution Network — Toy GCN Demo\n"
                 f"Ground truth: leaking at {truth_str}",
                 color="white", fontsize=13, y=1.01)

    plt.tight_layout()
    plt.savefig("gcn_leakage_result.png",
                dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    print("\n  📊 Plot saved → gcn_leakage_result.png")
    plt.close()


# ─────────────────────────────────────────────
# 7.  CONVERGENCE PLOT
# ─────────────────────────────────────────────
def plot_convergence(losses, accs):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    fig.patch.set_facecolor("#0f0f1a")
    epochs = range(1, len(losses) + 1)

    for ax in (ax1, ax2):
        ax.set_facecolor("#12122a")
        ax.tick_params(colors="white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#333355")

    ax1.plot(epochs, losses, color="#7ec8ff", linewidth=2)
    ax1.set_title("Training — CE Loss", color="white", fontweight="bold")
    ax1.set_xlabel("Epoch", color="white")
    ax1.set_ylabel("Cross-Entropy Loss", color="white")
    ax1.axhline(0.01, color="#ff8844", linestyle="--", alpha=0.7, label="convergence threshold")
    ax1.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=8)

    ax2.plot(epochs, [a * 100 for a in accs], color="#88ffbb", linewidth=2)
    ax2.set_title("Training — Accuracy", color="white", fontweight="bold")
    ax2.set_xlabel("Epoch", color="white")
    ax2.set_ylabel("ACC (%)", color="white")
    ax2.set_ylim(45, 102)

    plt.tight_layout()
    plt.savefig("gcn_training_curve.png",
                dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print("  📊 Training curve saved → gcn_training_curve.png")


# ─────────────────────────────────────────────
# 8.  MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("━"*55)
    print("  TOY GCN WATER LEAKAGE DETECTOR")
    print("  9-node grid city | 2 features/node | binary clf")
    print("━"*55)

    torch.manual_seed(42)
    random.seed(42)
    np.random.seed(42)

    # Build dataset & split
    print("\n[1/4] Generating synthetic hydraulic dataset ...")
    dataset  = build_dataset(n_normal=700, n_leak=700)
    split    = int(0.85 * len(dataset))
    train_ds = dataset[:split]
    test_ds  = dataset[split:]

    from torch_geometric.loader import DataLoader as PygLoader
    train_loader = PygLoader(train_ds, batch_size=32, shuffle=True)
    test_loader  = PygLoader(test_ds,  batch_size=64, shuffle=False)

    # Train
    print(f"\n[2/4] Training GCN ({len(train_ds)} graphs, 40 epochs max) ...")
    model = WaterLeakGCN(in_channels=2, hidden=32, out_channels=2)
    losses, accs = train_model(model, train_loader, n_epochs=40)
    plot_convergence(losses, accs)

    # Test set accuracy
    print("\n[3/4] Evaluating on test set ...")
    model.eval()
    correct, total = 0, 0
    tp, fp, tn, fn = 0, 0, 0, 0
    with torch.no_grad():
        for batch in test_loader:
            out, _ = model(batch.x, batch.edge_index, batch.batch)
            pred   = out.argmax(dim=1)
            correct += (pred == batch.y).sum().item()
            total   += batch.num_graphs
            tp += ((pred == 1) & (batch.y == 1)).sum().item()
            fp += ((pred == 1) & (batch.y == 0)).sum().item()
            tn += ((pred == 0) & (batch.y == 0)).sum().item()
            fn += ((pred == 0) & (batch.y == 1)).sum().item()

    acc = correct / total
    fnr = fn / (fn + tp + 1e-8)
    fpr = fp / (fp + tn + 1e-8)
    print(f"  Test ACC : {acc*100:.1f}%")
    print(f"  FNR      : {fnr*100:.1f}%  (leakage missed as normal)")
    print(f"  FPR      : {fpr*100:.1f}%  (normal flagged as leakage)")

    # Demo: "click the button"
    print("\n[4/4] Running demo scenarios ...")

    print("\n  Done! ✨  Check the two output PNGs.")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TOY GCN WATER LEAKAGE DETECTOR
  9-node grid city | 2 features/node | binary clf
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[1/4] Generating synthetic hydraulic dataset ...

[2/4] Training GCN (1190 graphs, 40 epochs max) ...
  Epoch   1 | CE loss: 0.7023 | ACC: 55.1%
  Epoch   5 | CE loss: 0.6231 | ACC: 64.2%
  Epoch  10 | CE loss: 0.4912 | ACC: 84.3%
  Epoch  15 | CE loss: 0.3443 | ACC: 92.4%
  Epoch  20 | CE loss: 0.2014 | ACC: 98.3%
  Epoch  25 | CE loss: 0.1181 | ACC: 99.5%
  Epoch  30 | CE loss: 0.1254 | ACC: 97.1%
  Epoch  35 | CE loss: 0.0554 | ACC: 100.0%
  Epoch  40 | CE loss: 0.0416 | ACC: 100.0%
  📊 Training curve saved → gcn_training_curve.png

[3/4] Evaluating on test set ...
  Test ACC : 100.0%
  FNR      : 0.0%  (leakage missed as normal)
  FPR      : 0.0%  (normal flagged as leakage)

[4/4] Running demo scenarios ...

  Done! ✨  Check the two output PNGs.


In [2]:

# Scenario A: leakage at node 4 (centre) + node 1
run_demo(model, leaking_nodes=[4, 1], noise_std=0.8)

# Scenario B: no leakage
run_demo(model, leaking_nodes=None, noise_std=0.8)



  SENSOR READING RESULT
  Ground truth : LEAKAGE at nodes [4, 1]
  Prediction   : ⚠️  LEAKAGE DETECTED
  P(normal)    : 0.0%
  P(leakage)   : 100.0%

  Node importance scores (higher = more suspicious):
    #1  Node 4  ████████████████████ 1.399
    #2  Node 7  ██████████████████ 1.326
    #3  Node 6  ██████████████████ 1.274
    #4  Node 8  ██████████████████ 1.274
    #5  Node 3  █████████████████ 1.197
    #6  Node 5  █████████████████ 1.197
    #7  Node 0  █████████████ 0.977
    #8  Node 2  █████████████ 0.977
    #9  Node 1  █████████████ 0.940

  📊 Plot saved → gcn_leakage_result.png

  SENSOR READING RESULT
  Ground truth : NORMAL
  Prediction   : ✅  NORMAL OPERATION
  P(normal)    : 97.3%
  P(leakage)   : 2.7%

  Node importance scores (higher = more suspicious):
    #1  Node 4  ████████████████████ 1.621
    #2  Node 1  █████████████████ 1.446
    #3  Node 3  █████████████████ 1.446
    #4  Node 5  █████████████████ 1.446
    #5  Node 7  █████████████████ 1.446
    #6  Node 

/tmp/ipykernel_48912/3449529327.py:358: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_48912/3449529327.py:359: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.savefig("gcn_leakage_result.png",



  📊 Plot saved → gcn_leakage_result.png
